# Exploratory Data Analysis — Shortage DSS (synthetic data)

Companion notebook to `run_project.py`. All data is **synthetic** (seeded generator).
Run the pipeline first: `python3 run_project.py` from the project root.

In [ ]:
import sqlite3, pandas as pd, matplotlib.pyplot as plt
con = sqlite3.connect('../data/processed/shortage_dss.db')
print('tables:', [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY 1")])

## 1. Demand intermittency (ADI / CV² — Syntetos & Boylan taxonomy)

In [ ]:
q = """
WITH d AS (
  SELECT material_id, txn_date AS dt, -quantity AS demand
  FROM inventory_transactions
  WHERE type='ISSUE' AND txn_date BETWEEN '2024-10-01' AND '2026-03-31'
),
agg AS (
  SELECT material_id,
         COUNT(*) AS n_days,
         SUM(demand) AS tot,
         AVG(demand) AS mu,
         SUM(demand*demand) AS sq
  FROM d GROUP BY 1
)
SELECT material_id, n_days, tot,
       CASE WHEN mu>0 THEN (sq/n_days - mu*mu)/(mu*mu) ELSE NULL END AS cv2
FROM agg WHERE tot > 0 ORDER BY cv2 DESC LIMIT 10
"""
pd.read_sql(q, con)

## 2. Supplier lead-time behaviour by tier

In [ ]:
q = """
SELECT s.tier,
       ROUND(AVG(julianday(gr.receipt_date) - julianday(pol.promised_date)),1) AS avg_late_days,
       ROUND(AVG(CASE WHEN julianday(gr.receipt_date) > julianday(pol.promised_date) THEN 1.0 ELSE 0 END),3) AS late_share
FROM goods_receipts gr
JOIN purchase_order_lines pol USING (po_line_id)
JOIN suppliers s ON pol.supplier_id = s.supplier_id
GROUP BY 1
"""
df = pd.read_sql(q, con); df

## 3. Shortage flags over time

In [ ]:
df = pd.read_sql("SELECT week_start, COUNT(*) AS n_flags FROM v_shortage_flags GROUP BY 1 ORDER BY 1", con)
df['week_start'] = pd.to_datetime(df['week_start'])
df.plot(x='week_start', y='n_flags', title='Shortage flags per week (synthetic)')
plt.show()